# 使用表示模型进行文本分类


## 1.使用预训练的特定任务模型（ TWITTER 评论）进行电影影评的情感分类

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
from datasets import load_dataset


In [ ]:
# 影评数据
data = load_dataset("rotten_tomatoes", cache_dir="./cache")

data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [ ]:
# 查看训练数据的第0， 1， 最后一个元素
data['train'][0, 1, -1]

{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
  'the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson\'s expanded vision of j . r . r . tolkien\'s middle-earth .',
  'things really get weird , though not particularly scary : the movie is all portent and no content .'],
 'label': [1, 1, 0]}

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Labels: 0 -> Negative; 1 -> Neutral; 2 -> Positive
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

tokenizer = AutoTokenizer.from_pretrained(model_path, cache_dir="./model_cache")
# Text Classification	    AutoModelForSequenceClassification	情感分析、垃圾邮件识别、主题分类
# Causal Language Modeling	AutoModelForCausalLM	            文本续写、对话生成、代码生成（GPT/LLaMA 类）
# Masked Language Modeling	AutoModelForMaskedLM	            掩码填充、语义理解（BERT 类）
# Seq2Seq Language Modeling	AutoModelForSeq2SeqLM           	机器翻译、文本摘要、改写（T5/BART 类）
# Token Classification	    AutoModelForTokenClassification	    命名实体识别（NER）、词性标注
# Question Answering	    AutoModelForQuestionAnswering	    抽取式问答（如 SQuAD）
# Feature Extraction	    AutoModel	                        提取文本语义向量（无任务头）
model = AutoModelForSequenceClassification.from_pretrained(model_path, cache_dir="./model_cache")

pipe = pipeline(model = model_path, tokenizer = model_path, top_k = 2)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassifi

In [ ]:
# 结果是按照从大到小的顺序排列的
pipe(["I do not like this movie"])[0]

[{'label': 'negative', 'score': 0.9120093584060669},
 {'label': 'neutral', 'score': 0.07591026276350021}]

In [ ]:
# 结果是按照从大到小的顺序排列的
pipe(["I love this movie"])[0]

[{'label': 'positive', 'score': 0.9674931764602661},
 {'label': 'neutral', 'score': 0.027071893215179443}]

In [ ]:
from transformers.pipelines.pt_utils import KeyDataset
import numpy as np
# 进度条
from tqdm import tqdm

In [ ]:
y_pred = []
# KeyDataset: 从 dataset 中获取特定的 key
for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
    label = output[0]["label"]
    if label == "neutral" :
        label = output[1]["label"]
    assignment = 0
    if label == "positive":
        assignment = 1
    y_pred.append(assignment)

 88%|████████▊ | 943/1066 [00:32<00:03, 32.22it/s]

In [ ]:
from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred):
    performance = classification_report(y_true, y_pred, target_names=["negative", "positive"])
    print(performance)

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)

              precision    recall  f1-score   support

    negative       0.76      0.88      0.81       533
    positive       0.86      0.72      0.78       533

    accuracy                           0.80      1066
   macro avg       0.81      0.80      0.80      1066
weighted avg       0.81      0.80      0.80      1066



## 2. 使用嵌入式模型和适配网络进行文本分类

In [2]:
import os
from sentence_transformers import SentenceTransformer

In [3]:
from datasets import load_dataset
data = load_dataset("rotten_tomatoes", cache_dir="./cache")

In [4]:
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', cache_folder="./model_cache")

In [5]:
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)
# 768维嵌入向量
train_embeddings.shape

Batches:   0%|          | 0/267 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

(8530, 768)

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred):
    performance = classification_report(y_true, y_pred, target_names=["negative", "positive"])
    print(performance)

In [6]:
clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data["train"]["label"])

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [10]:
y_pred = clf.predict(test_embeddings)
evaluate_performance(data["test"]["label"], y_pred)

              precision    recall  f1-score   support

    negative       0.85      0.86      0.85       533
    positive       0.86      0.85      0.85       533

    accuracy                           0.85      1066
   macro avg       0.85      0.85      0.85      1066
weighted avg       0.85      0.85      0.85      1066



## 3. 零样本嵌入式预测

对类别进行描述，如“一条负面影评”，通过嵌入模型生成嵌入向量，然后对文档生成嵌入向量，比较余弦相似度。

In [11]:
# 描述信息的定义具有决定性作用
label_embeddings = model.encode(["A very negative review", "A very positive review"])
#label_embeddings = model.encode(["A very negative review", "A very positive review"])

In [14]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
# 为每个文档找到最匹配的标签
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

In [13]:
evaluate_performance(data["test"]["label"], y_pred)

              precision    recall  f1-score   support

    negative       0.73      0.84      0.78       533
    positive       0.81      0.69      0.75       533

    accuracy                           0.76      1066
   macro avg       0.77      0.76      0.76      1066
weighted avg       0.77      0.76      0.76      1066

